# Music Annotation Project
## Notebook 73R: V8.8 Retained-Residue Routing and Assembly Audit

**Runtime: CPU. Read-only diagnostic.** Run after the completed Notebook 72 prediction lock.

This notebook replaces the failed treatment-strength counterfactual. It does **not** tune thresholds, change SAFE_REMOVE, generate a new correction policy or rerun learned inference.

### Why this audit exists
The historical V8.8 operational contract is:

```text
M5_REMOVE = M3 & ~SAFE_REMOVE
reconstructed = reconstruct_residual(observed, M5_REMOVE)
out = reconstructed
out[SAFE_REMOVE] = paper[SAFE_REMOVE]
```

So SAFE_REMOVE is a set of pixels that receive direct paper replacement and are removed from the residual inpainting mask. Pixels that are **not** SAFE_REMOVE remain in `M5_REMOVE` and are reconstructed through the residual-mask path.

The current unified `v8_8_operator.py` must therefore be audited against that historical assembly contract before any new cleanup experiment is designed.

### Questions answered
For all 151 Notebook 72 pages:

1. Does the saved Notebook 72 `corrected.png` equal the current operator's assembly semantics?
2. What would the historical Notebook 53 M5_REMOVE assembly produce from the *same frozen* M3 and SAFE_REMOVE maps?
3. Where does visible retained darkness sit: SAFE_REMOVE, M5_REMOVE/fallback, V7.8 preserve, strict PRINT protection, V7.8 remove, or V7.8 ambiguous?
4. How much retained darkness is actually attributable to assembly semantics rather than model selection?
5. Which pages have the largest difference between the current output and the historical M5_REMOVE assembly?

No clean page or truth-derived data is read.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, zipfile
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
SOURCE_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook73r_v8_8_retained_residue_routing_audit'
CROP_MARGIN=48
TOP_PAGES=20

REQUIRED=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','pages']
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NOTEBOOK72_ROOT,REQUIRED)
if SOURCE is None: raise RuntimeError('No complete Notebook 72 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook73r_retained_residue_audit'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; PAGES=OUT/'pages'; VISUAL=OUT/'visuals'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,PAGES,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Source:',SOURCE); print('Output:',OUT)

## 2. Integrity helpers and closed-lock verification

In [ ]:
def require(x,msg):
    if not bool(x): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp'); t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); t.replace(path)
def read_gray(path):
    x=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(x is not None,f'Unreadable image: {path}'); return x

def padded_components(mask,margin=CROP_MARGIN,min_area=8):
    n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=min_area: boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def reconstruct_residual(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1]; mk=(mask[y0:y1,x0:x1].astype(np.uint8)*255)
        cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA); local=result[y0:y1,x0:x1]; local[mk>0]=cand[mk>0]; result[y0:y1,x0:x1]=local
    out=observed.copy(); out[mask]=result[mask]
    require(np.array_equal(out[~mask],observed[~mask]),'Residual reconstruction changed outside mask.')
    return out

lock=json.loads((SOURCE/'prediction_lock.json').read_text()); verdict=json.loads((SOURCE/'notebook72_verdict.json').read_text())
require(lock.get('state')=='CLOSED','Notebook 72 prediction lock is not closed.')
require(verdict.get('first_failed_gate') is None,'Notebook 72 has a failed gate.')
roster=pd.read_csv(SOURCE/'inputs/runtime_roster.csv',dtype=str).fillna(''); hashes=pd.read_csv(SOURCE/'tables/page_output_hashes.csv',dtype=str).fillna('').set_index('page_key')
require(lock['payload']['runtime_roster_sha256']==sha256_file(SOURCE/'inputs/runtime_roster.csv'),'Roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(SOURCE/'tables/page_output_hashes.csv'),'Output-hash table changed after lock.')
require(len(roster)==len(hashes),'Roster/hash count mismatch.')
source_identity={'created_utc':datetime.now(timezone.utc).isoformat(),'source_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'operator_bundle_id':lock['payload']['operator_bundle_id'],'pages':len(roster),'clean_truth_read':False}
atomic_json(INPUTS/'source_identity.json',source_identity); print(json.dumps(source_identity,indent=2))

## 3. Reconstruct decision/routing maps from locked Notebook 72 artefacts

In [ ]:
page_state={}; integrity=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; d=SOURCE/'pages'/key
    observed=read_gray(d/'observed.png'); current=read_gray(d/'corrected.png'); baseline=read_gray(d/'baseline_v78.png'); prepared=read_gray(d/'prepared_mask.png')>0; safe=read_gray(d/'safe_remove.png')>0; m3=read_gray(d/'m3.png')>0; m5_saved=read_gray(d/'m5_remove.png')>0; reason=read_gray(d/'reason_map.png'); features=pd.read_parquet(d/'observable_features.parquet')
    h=hashes.loc[key]
    require(array_hash(observed)==h.observed_array_sha256,f'{key}: observed hash mismatch')
    require(array_hash(current)==h.corrected_array_sha256,f'{key}: current corrected hash mismatch')
    require(array_hash(baseline)==h.baseline_array_sha256,f'{key}: baseline hash mismatch')
    require(array_hash(safe)==h.safe_remove_array_sha256,f'{key}: safe hash mismatch')
    require(array_hash(m3)==h.m3_array_sha256,f'{key}: m3 hash mismatch')
    require(array_hash(m5_saved)==h.m5_remove_array_sha256,f'{key}: m5 hash mismatch')
    require(not np.any(safe&~m3),f'{key}: SAFE_REMOVE outside M3')
    expected_m5=m3&~safe
    require(np.array_equal(expected_m5,m5_saved),f'{key}: saved M5_REMOVE != M3 & ~SAFE_REMOVE')
    # reason-map contract emitted by the current unified operator: 1 preserve, 2 strict PRINT, 3 M3 fallback/not-safe-remove, 4 SAFE_REMOVE
    preserve=reason==1; strict_print=reason==2; fallback=reason==3; safe_reason=reason==4
    require(np.array_equal(safe,safe_reason),f'{key}: reason SAFE_REMOVE mismatch')
    require(np.array_equal(m5_saved,fallback),f'{key}: reason fallback/M5 mismatch')
    # Recover V7.8 route context at M3 rows from the regression-proven feature table.
    fmap={name:np.zeros(observed.shape,np.float32) for name in ['paper_value','p_print','p_remove','local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction']}
    yy=features.y.astype(int).to_numpy(); xx=features.x.astype(int).to_numpy()
    for name in fmap: fmap[name][yy,xx]=pd.to_numeric(features[name],errors='coerce').fillna(np.nan).to_numpy(np.float32)
    page_state[key]={'page':str(r.page),'population':str(r.population),'observed':observed,'current':current,'baseline':baseline,'prepared':prepared,'safe':safe,'m3':m3,'m5':m5_saved,'preserve':preserve,'strict_print':strict_print,'features':features,'maps':fmap}
    integrity.append({'page_key':key,'m5_equals_m3_minus_safe':True,'safe_subset_m3':True,'outside_current_drift':int(np.sum(current[~prepared]!=observed[~prepared])),'features_rows':len(features)})
pd.DataFrame(integrity).to_csv(AUDIT/'locked_state_integrity.csv',index=False)
print('Verified locked pages:',len(page_state))

## 4. Historical Notebook 53 M5_REMOVE assembly on the exact frozen decisions

In [ ]:
assembly_rows=[]
for key,s in page_state.items():
    observed=s['observed']; m3=s['m3']; safe=s['safe']; m5=s['m5']; features=s['features']
    paper=np.full(observed.shape,np.nan,np.float32); yy=features.y.astype(int).to_numpy(); xx=features.x.astype(int).to_numpy(); paper[yy,xx]=pd.to_numeric(features.paper_value,errors='raise').to_numpy(np.float32)
    require(np.isfinite(paper[safe]).all(),f'{key}: SAFE_REMOVE missing paper values')
    reconstructed=reconstruct_residual(observed,m5)
    historical=reconstructed.copy(); historical[safe]=np.clip(np.rint(paper[safe]),0,255).astype(np.uint8)
    require(np.array_equal(historical[~m3],observed[~m3]),f'{key}: historical assembly changed outside M3')
    current=s['current']; baseline=s['baseline']; diff=current!=historical
    d=PAGES/key; d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'current_notebook72.png'),current); cv2.imwrite(str(d/'historical_m5_assembly.png'),historical); cv2.imwrite(str(d/'difference_current_vs_historical.png'),diff.astype(np.uint8)*255); cv2.imwrite(str(d/'M3.png'),m3.astype(np.uint8)*255); cv2.imwrite(str(d/'SAFE_REMOVE.png'),safe.astype(np.uint8)*255); cv2.imwrite(str(d/'M5_REMOVE.png'),m5.astype(np.uint8)*255)
    assembly_rows.append({'page_key':key,'population':s['population'],'m3_pixels':int(m3.sum()),'safe_remove_pixels':int(safe.sum()),'m5_remove_pixels':int(m5.sum()),'current_vs_historical_different_pixels':int(diff.sum()),'different_inside_safe_remove':int(np.sum(diff&safe)),'different_inside_m5_remove':int(np.sum(diff&m5)),'current_equals_historical':bool(not diff.any()),'current_array_sha256':array_hash(current),'historical_array_sha256':array_hash(historical)})
assembly=pd.DataFrame(assembly_rows); assembly.to_csv(TABLES/'current_vs_historical_m5_assembly.csv',index=False)
print(assembly.groupby('population').agg(pages=('page_key','size'),different_pixels=('current_vs_historical_different_pixels','sum'),pages_different=('current_equals_historical',lambda x:int((~x).sum()))))

## 5. Partition retained darkness by exact runtime disposition and V7.8 route evidence

In [ ]:
pixel_rows=[]; page_rows=[]
for key,s in page_state.items():
    obs=s['observed']; current=s['current']; baseline=s['baseline']; prepared=s['prepared']; safe=s['safe']; m3=s['m3']; m5=s['m5']; preserve=s['preserve']; strict=s['strict_print']; f=s['features']
    # Visible residual darkness relative to frozen local paper estimate is available on every M3 feature row.
    paper=np.full(obs.shape,np.nan,np.float32); yy=f.y.astype(int).to_numpy(); xx=f.x.astype(int).to_numpy(); paper[yy,xx]=pd.to_numeric(f.paper_value,errors='raise').to_numpy(np.float32)
    p_remove=np.full(obs.shape,np.nan,np.float32); p_remove[yy,xx]=pd.to_numeric(f.p_remove,errors='coerce').to_numpy(np.float32)
    p_print=np.full(obs.shape,np.nan,np.float32); p_print[yy,xx]=pd.to_numeric(f.p_print,errors='coerce').to_numpy(np.float32)
    # Route classification from the exact approved observable builder.
    route=np.full(obs.shape,'OUTSIDE_M3',object)
    lp=pd.to_numeric(f.local_preserve_fraction,errors='coerce').fillna(0).to_numpy(); lr=pd.to_numeric(f.local_remove_fraction,errors='coerce').fillna(0).to_numpy(); la=pd.to_numeric(f.local_ambiguous_fraction,errors='coerce').fillna(0).to_numpy()
    dominant=np.where((lr>=lp)&(lr>=la),'V78_REMOVE_CONTEXT',np.where((la>=lp)&(la>=lr),'V78_AMBIGUOUS_CONTEXT','V78_PRESERVE_CONTEXT')); route[yy,xx]=dominant
    residual=np.maximum(paper-current.astype(np.float32),0); valid=m3&np.isfinite(paper)
    disposition=np.full(obs.shape,'OUTSIDE_M3',object); disposition[m5]='M5_REMOVE_RECONSTRUCT'; disposition[safe]='SAFE_REMOVE_PAPER'; disposition[preserve]='V78_PRESERVE'; disposition[strict]='STRICT_PRINT_PROTECT'
    for name,mask in [('SAFE_REMOVE_PAPER',safe),('M5_REMOVE_RECONSTRUCT',m5),('V78_PRESERVE',preserve),('STRICT_PRINT_PROTECT',strict)]:
        q=mask&np.isfinite(paper)
        if not q.any(): continue
        vals=residual[q]
        page_rows.append({'page_key':key,'population':s['population'],'disposition':name,'pixels_with_paper':int(q.sum()),'residual_10plus':int((vals>=10).sum()),'residual_20plus':int((vals>=20).sum()),'residual_40plus':int((vals>=40).sum()),'mean_residual':float(vals.mean()),'p90_residual':float(np.percentile(vals,90)),'residual_mass':float(vals.sum())})
    ys,xs=np.where(valid&(residual>=10))
    for y,x in zip(ys,xs):
        pixel_rows.append({'page_key':key,'population':s['population'],'y':int(y),'x':int(x),'observed':int(obs[y,x]),'current':int(current[y,x]),'baseline_v78':int(baseline[y,x]),'paper_value':float(paper[y,x]),'residual_darkness':float(residual[y,x]),'disposition':str(disposition[y,x]),'route_context':str(route[y,x]),'p_remove':float(p_remove[y,x]),'p_print':float(p_print[y,x]),'safe_remove':bool(safe[y,x]),'m5_remove':bool(m5[y,x])})
retained=pd.DataFrame(pixel_rows); disposition_pages=pd.DataFrame(page_rows)
retained.to_parquet(TABLES/'retained_dark_pixels_10plus.parquet',index=False); disposition_pages.to_csv(TABLES/'residual_by_disposition_and_page.csv',index=False)
summary=(retained.groupby(['disposition','route_context'],as_index=False).agg(pixels=('residual_darkness','size'),pages=('page_key','nunique'),source_groups=('population','nunique'),mean_residual=('residual_darkness','mean'),p90_residual=('residual_darkness',lambda x:float(np.percentile(x,90))),residual_mass=('residual_darkness','sum'),mean_p_remove=('p_remove','mean'),mean_p_print=('p_print','mean')).sort_values('residual_mass',ascending=False))
summary.to_csv(TABLES/'retained_residue_reason_summary.csv',index=False); print(summary.to_string(index=False))

## 6. Diagnostic overlays and highest-opportunity pages

In [ ]:
def overlay(gray,layers,alpha=.58):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); out=base.copy()
    for mask,bgr in layers:
        colour=np.zeros_like(base); colour[:]=bgr; out[mask]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[mask]
    return out
# Red=M5 residual reconstruction, green=SAFE_REMOVE paper, blue=strict PRINT, orange=V7.8 preserve.
page_scores=[]
for key,s in page_state.items():
    f=s['features']; paper=np.full(s['current'].shape,np.nan,np.float32); yy=f.y.astype(int).to_numpy(); xx=f.x.astype(int).to_numpy(); paper[yy,xx]=f.paper_value.to_numpy(np.float32); residual=np.maximum(paper-s['current'].astype(np.float32),0); hot=np.isfinite(paper)&(residual>=20)
    layers=[(s['m5']&hot,(0,0,255)),(s['safe']&hot,(0,200,0)),(s['strict_print']&hot,(255,0,0)),(s['preserve']&hot,(0,165,255))]
    image=overlay(s['current'],layers); cv2.imwrite(str(PAGES/key/'retained_residue_overlay.png'),image)
    page_scores.append({'page_key':key,'population':s['population'],'residual20_pixels':int(hot.sum()),'m5_residual20':int(np.sum(hot&s['m5'])),'safe_residual20':int(np.sum(hot&s['safe'])),'strict_print_residual20':int(np.sum(hot&s['strict_print'])),'preserve_residual20':int(np.sum(hot&s['preserve'])),'residual_mass':float(residual[hot].sum())})
page_scores=pd.DataFrame(page_scores).sort_values('residual_mass',ascending=False); page_scores.to_csv(TABLES/'page_residue_ranking.csv',index=False)

def sheet(rows,name):
    rows=rows.head(TOP_PAGES)
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),3,figsize=(15,max(5,5.5*len(rows))),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        key=r.page_key
        imgs=[read_gray(SOURCE/'pages'/key/'corrected.png'),read_gray(PAGES/key/'historical_m5_assembly.png'),cv2.cvtColor(cv2.imread(str(PAGES/key/'retained_residue_overlay.png')),cv2.COLOR_BGR2RGB)]
        titles=[f'{key} current Notebook 72',f'{key} historical M5 assembly',f'{key} retained-residue reasons']
        for j,(im,title) in enumerate(zip(imgs,titles)):
            axes[i,j].imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0 if im.ndim==2 else None,vmax=255 if im.ndim==2 else None); axes[i,j].set_title(title); axes[i,j].axis('off')
    plt.tight_layout(); plt.savefig(VISUAL/f'{name}.png',dpi=125,bbox_inches='tight'); plt.close(fig)
sheet(page_scores,'highest_retained_residue_pages')
sheet(assembly.sort_values('current_vs_historical_different_pixels',ascending=False),'largest_current_vs_historical_assembly_differences')
print(page_scores.head(20).to_string(index=False))

## 7. Fail-closed interpretation and audit verdict

In [ ]:
total_diff=int(assembly.current_vs_historical_different_pixels.sum()); pages_diff=int((~assembly.current_equals_historical).sum())
residual_total=float(summary.residual_mass.sum()) if len(summary) else 0.0
m5_mass=float(summary.loc[summary.disposition=='M5_REMOVE_RECONSTRUCT','residual_mass'].sum()) if len(summary) else 0.0
safe_mass=float(summary.loc[summary.disposition=='SAFE_REMOVE_PAPER','residual_mass'].sum()) if len(summary) else 0.0
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':'PASS_DIAGNOSTIC_AUDIT_COMPLETE','promotion_authority':False,'prediction_lock_id':lock['lock_id'],'pages':len(roster),'current_vs_historical_assembly':{'pages_different':pages_diff,'different_pixels':total_diff,'all_equal':pages_diff==0},'retained_darkness_10plus':{'pixels':len(retained),'residual_mass':residual_total,'m5_remove_reconstruct_mass':m5_mass,'safe_remove_paper_mass':safe_mass,'m5_share_of_reported_mass':m5_mass/residual_total if residual_total else None},'interpretation':('Current Notebook 72 assembly differs from the historical Notebook 53 M5_REMOVE contract; correct implementation equivalence before designing another cleanup challenger.' if pages_diff else 'Current Notebook 72 assembly matches historical M5_REMOVE semantics; retained residue is therefore attributable to frozen routing/selection rather than an assembly mismatch.'),'clean_truth_read':False}
atomic_json(OUT/'notebook73r_verdict.json',record)
lines=['NOTEBOOK 73R — RETAINED-RESIDUE ROUTING AND ASSEMBLY AUDIT','',f'Verdict: {record["verdict"]}',f'Pages: {len(roster)}',f'Pages where current output differs from historical M5 assembly: {pages_diff}',f'Total current-vs-historical different pixels: {total_diff}',f'Retained-dark pixels (>=10 below local paper, inside M3): {len(retained)}',f'Retained residual mass attributed to M5_REMOVE reconstruction: {m5_mass:.0f}',f'Retained residual mass attributed to SAFE_REMOVE paper route: {safe_mass:.0f}','','INTERPRETATION',record['interpretation'],'','No clean truth was read. No model, probability, threshold or decision map was changed.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines)); print(chr(10).join(lines))

## 8. Curated upload package

In [ ]:
required=[OUT/'executive_summary.txt',OUT/'notebook73r_verdict.json',INPUTS/'source_identity.json',AUDIT/'locked_state_integrity.csv',TABLES/'current_vs_historical_m5_assembly.csv',TABLES/'residual_by_disposition_and_page.csv',TABLES/'retained_residue_reason_summary.csv',TABLES/'page_residue_ranking.csv',VISUAL/'highest_retained_residue_pages.png',VISUAL/'largest_current_vs_historical_assembly_differences.png']
for p in required: require(p.exists(),f'Missing package item: {p}')
manifest=[]
for src in required:
    dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'README.txt').write_text('Notebook 73R read-only V8.8 retained-residue routing and historical assembly audit. Start with executive_summary.txt and notebook73r_verdict.json. No production promotion or model changes.'+chr(10))
with zipfile.ZipFile(UPLOAD/'notebook73r_retained_residue_audit_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip': z.write(p,p.name)
print('Upload bundle:',UPLOAD/'notebook73r_retained_residue_audit_bundle.zip')

## Decision tree after Notebook 73R

- **Current Notebook 72 output differs materially from historical M5 assembly:** fix `v8_8_operator.py`, rerun equivalence, and treat the existing 151-page Notebook 72 output as an integration experiment rather than frozen V8.8 semantics.
- **Assembly matches, and retained residue is mostly M5_REMOVE reconstruction:** investigate selection/routing and reconstruction only within the already-authorised M5 residual path.
- **Assembly matches, and residue is mostly strict PRINT/V7.8 preserve:** investigate false protection or conservative print-preservation coverage, not REMOVE treatment strength.
- **Assembly matches, and SAFE_REMOVE still carries substantial residual darkness:** investigate a genuine direct-paper/tone implementation issue.

Do not change a threshold until this attribution is known.